# Data preprocessing for master-oez61

In [ ]:
!rm -r data/
!rm -r __MACOSX/
!rm -r data_val/

In [ ]:
!mkdir data
!unzip data.zip -d ./data

In [ ]:
import yaml
import os

# Modify original data label
def update_original_data(yaml_path, labels_paths):
    yaml_data = {}
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)

    labels = yaml_data['names']
    for path in labels_paths:
        for file in os.listdir(path):
            new_content = []
            # Read label files
            with open(f'{path}/{file}', 'r') as f:
                content = f.readlines()
                for line in content:
                    # Skip 1s and 5z, as this dataset is using richii mahjong
                    # idx = int(line.split()[0])
                    # if labels[idx] in ['1s', '5z']:
                    #     continue

                    # update the class id to 0, as only 1 class
                    words = line.split(' ')[1:]
                    new_words = ['0'] + words
                    new_line = ' '.join(new_words)
                    new_content.append(new_line)


            # Write label files
            with open(f'{path}/{file}', 'w') as f:
                f.writelines(new_content)
                f.close()

In [ ]:
from os.path import split
import os

# Filter out malformatted files
# For yolo without OBB, each line in label file contains only 5 values
def malformat_filter(paths):
    for path in paths:
        targets = []
        for file in os.listdir(path+'labels'):
            lines = []
            label_path = path+'labels/'+file
            with open(label_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                words = line.split(' ')
                if len(words) > 5:
                    targets.append('.'.join(label_path.split('.')[:-1]))
                    break

        for target in targets:
            os.remove(target+'.txt')
            os.remove(target.replace('labels', 'images', 1)+'.jpg')

In [ ]:
# Modify original yaml to accomodate Hong Kong Mahjong Tiles
import yaml
import os

def update_original_yaml(path):
    yaml_content = {}
    with open(path, 'r') as f:
        yaml_content = yaml.safe_load(f)
        print(yaml_content)
        f.close()

    yaml_content['path'] = './data'
    yaml_content['test'] = 'test/images'
    yaml_content['train'] = 'train/images'
    yaml_content['val'] = 'valid/images'
    yaml_content['names'] = ['tile']
    yaml_content['nc'] = len(yaml_content['names'])

    with open('yolo11.yaml', 'w') as f:
        yaml.safe_dump(yaml_content, f)

In [ ]:
# Update data/train/labels
labels_paths = ['data/train/labels', 'data/valid/labels']
update_original_data('data/data.yaml', labels_paths)
malformat_filter(['data/train/', 'data/valid/', 'data/test/'])

# Data preprocessing for mahjong-tiles-non-riichi-xe0ya
- Copies '1s' and '5z' to main dataset 'data/'

In [ ]:
!rm -r data1/
!rm -r __MACOSX/

In [ ]:
!mkdir data1/
!unzip data_cls.zip -d data1/

In [ ]:
# Traverse all train + val (image, label) pairs
# If a line in label file is in ['1s', '5z'], update to 'tile' and copy it to 'data'

import shutil
import os
def update_data_copy(path, target):
    train_path = os.path.join(path, 'train')
    valid_path = os.path.join(path, 'valid')
    
    yaml_data = {}
    with open(os.path.join(path, 'data.yaml') , 'r') as f:
        yaml_data = yaml.safe_load(f)

    labels = yaml_data['names']

    # Traverse all label files in train
    train_label_path = os.path.join(train_path, 'labels/')
    for label_file in os.listdir(train_label_path):
        file_content = []
        new_content = []
        with open(os.path.join(train_label_path, label_file), 'r') as f:
            file_content = f.readlines()

        for line in file_content:
            # idx = int(line.split()[0])
            # label_name = labels[idx]
            # if label_name in ['s1', 'z5']:
            words = line.split()[1:]
            words = ['0'] + words
            new_content.append(words)

        if new_content:
            text_content = []
            for line in new_content:
                text_content.append(' '.join(line) + '\n')
            
            with open(os.path.join(train_label_path, label_file), 'w') as f:
                f.writelines(text_content)
            
            # Copy to data
            train_image_path = os.path.join(train_path, 'images')
            img_path = os.path.join(train_image_path, label_file[:-4] + '.jpg')
            print(img_path)
        else: 
            # remove those files
            train_image_path = os.path.join(train_path, 'images')
            os.remove(os.path.join(train_image_path, label_file[:-4] + '.jpg'))
            os.remove(os.path.join(train_label_path, label_file))
    
    # Copy all path/train/(images, labels) to target/train/(image, labels)
    train_image_path = os.path.join(train_path, 'images/')
    os.makedirs(os.path.join(target, 'train', 'images'), exist_ok=True)
    os.makedirs(os.path.join(target, 'train', 'labels'), exist_ok=True)
    for f in os.listdir(train_image_path):
        shutil.copy2(os.path.join(train_image_path, f), os.path.join(target, 'train', 'images'))

    for f in os.listdir(train_label_path):
        shutil.copy2(os.path.join(train_label_path, f), os.path.join(target, 'train', 'labels'))


    # Traverse all label files in valid
    valid_label_path = os.path.join(valid_path, 'labels')
    for label_file in os.listdir(valid_label_path):
        file_content = []
        new_content = []
        with open(os.path.join(valid_label_path, label_file), 'r') as f:
            file_content = f.readlines()

        for line in file_content:
            # idx = int(line.split()[0])
            # label_name = labels[idx]
            # if label_name in ['s1', 'z5']:
            words = line.split()[1:]
            words = ['0'] + words
            new_content.append(words)

        if new_content:
            text_content = []
            for line in new_content:
                text_content.append(' '.join(line) + '\n')
            
            with open(os.path.join(valid_label_path, label_file), 'w') as f:
                f.writelines(text_content)
            
            # Copy to data
            valid_image_path = os.path.join(valid_path, 'images')
            img_path = os.path.join(valid_image_path, label_file[:-4] + '.jpg')
            print(img_path)
        else: 
            # remove those files
            valid_image_path = os.path.join(valid_path, 'images')
            os.remove(os.path.join(valid_image_path, label_file[:-4] + '.jpg'))
            os.remove(os.path.join(valid_label_path, label_file))

    # Copy all path/train/(images, labels) to target/train/(image, labels)
    valid_image_path = os.path.join(valid_path, 'images/')
    os.makedirs(os.path.join(target, 'valid', 'images'), exist_ok=True)
    os.makedirs(os.path.join(target, 'valid', 'labels'), exist_ok=True)
    for f in os.listdir(valid_image_path):
        shutil.copy2(os.path.join(valid_image_path, f), os.path.join(target, 'valid', 'images'))

    for f in os.listdir(valid_label_path):
        shutil.copy2(os.path.join(valid_label_path, f), os.path.join(target, 'valid', 'labels'))


In [ ]:
malformat_filter(['data1/train/', 'data1/valid/', 'data1/test/'])
update_data_copy('./data1', './data')

In [ ]:
update_original_yaml('./data/data.yaml')

In [ ]:
import cv2
from datetime import datetime

# Create cropped image dataset for verification
def crop_val_set(path, target):
    os.makedirs(target, exist_ok=True)
    img_dir_path = os.path.join(path, 'images')
    label_dir_path = os.path.join(path, 'labels')
    for img_file_name in os.listdir(img_dir_path):
        label_file_name = img_file_name[:-4] + '.txt'
        img = cv2.imread(os.path.join(img_dir_path, img_file_name))
        h, w = img.shape[:2]
        with open(os.path.join(label_dir_path, label_file_name), 'r') as f:
            tiles = f.readlines()
            for tile in tiles:
                # print(tile)
                if not tile:
                    continue

                cls_id, x_c, y_c, bw, bh = map(float, tile.strip().split()[:5])
                cls_id = int(cls_id)

                # Denormalize
                x1 = int((x_c - bw/2) * w)
                y1 = int((y_c - bh/2) * h)
                x2 = int((x_c + bw/2) * w)
                y2 = int((y_c + bh/2) * h)

                # Clamp
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                crop = img[y1:y2, x1:x2]

                current_time = datetime.now()
                crop_path = os.path.join(target, str(current_time) + '.jpg')
                cv2.imwrite(crop_path, crop)
                print(f"Saved: {crop_path}")

In [ ]:
crop_val_set('data/train', 'data_val')
crop_val_set('data/valid', 'data_val')